# G7b — Does B1 improve when it is told *whose* face is whose?

G6b/G6c (train+val, out-of-fold over 45 episodes) showed that reading the **listener's** face in clip III forecasts
B better than reading A's face (zero-shot ΔUAR +3.72 [+0.60, +7.24]), that the gain comes from the expression and
not from whether a reaction shot exists, and that B's face is often already present in clips I/II.
B1 sees whole frames and has no notion of who is who. This notebook adds the G6b **role-grounded face features**
(HSEmotion 8 probabilities + valence/arousal + presence + frame share, per role) to B1.

| Arm | Face input (per role: A, listener in III, listener seen in I/II, dominant face of II, of I) |
|---|---|
| `B1` | none, identical to G3b's B1 (reproduction check) |
| `B1+faces` | all 5 roles × 12 = 60 numbers |
| `B1+faces_early` | same, but the listener uses only frames in the first 80% of clip III (secondary) |
| `B1+presence` | control: only presence + frame share per role, no expression |

Fusion: the face vector (z-scored with training statistics) goes through a small MLP whose **last layer starts at
zero** and is added to B1's pooled representation, so every face arm starts exactly as B1.
Both selection protocols, 5 seeds, paired episode bootstrap against B1. **Test stays locked** (the face CSV has
train+val only).

Gate: `B1+faces` − `B1` (inner-dev selection) ΔUAR > 0 with CI lower bound > 0 and ≥ 3/5 seed wins; under
val selection `B1+faces@val` ≥ `B1@val`; `B1+presence` does not explain the gain.

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
ROLE_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/role-features/g6b_role_features.csv",
                          "/kaggle/input/role-features/g6b_role_features.csv")
OUT_DIR = "/kaggle/working"

SEEDS = [42, 123, 456, 789, 1024]
N_FOLDS = 5
N_INNER_DEV_SOURCES = 5
REC_EPOCHS, FC_EPOCHS, PATIENCE = 60, 50, 8
REC_BATCH, FC_BATCH = 64, 32
LR, WEIGHT_DECAY = 1e-4, 1e-5
POL_WEIGHT = 0.3
CERT_WEIGHTS = {'1': 1.0, '2': 0.75, '3': 0.5}   # bookkeeping only
REC_SEED = 42

# (name, face set, selection protocol)
EXPERIMENTS = [
    ("B1",                 None,       "inner_dev"),
    ("B1+faces",           "full",     "inner_dev"),
    ("B1+faces_early",     "early",    "inner_dev"),
    ("B1+presence",        "presence", "inner_dev"),
    ("B1@val",             None,       "val"),
    ("B1+faces@val",       "full",     "val"),
    ("B1+faces_early@val", "early",    "val"),
    ("B1+presence@val",    "presence", "val"),
]
G3B_B1_SEED_MEAN = {"inner_dev": 21.65, "val": 25.77}   # G3b per-seed mean UAR of B1, for the reproduction check
EVAL_SPLIT = "val"
UNLOCK_TEST = False

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF", "annotation.csv")
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
train_all = sp[sp.split == 'train'].reset_index(drop=True)
ev = eval_rows(sp, EVAL_SPLIT, UNLOCK_TEST)

train_sources = sorted(train_all.source_folder.unique())
rng = random.Random(0)
shuffled = train_sources[:]
rng.shuffle(shuffled)
FOLD_OF = {s: i % N_FOLDS for i, s in enumerate(shuffled)}
inner_dev_sources = sorted(random.Random(1).sample(train_sources, N_INNER_DEV_SOURCES))

lab = ann[ann[7].notna()].copy()
lab['ep'] = [c.split('/')[0] for c in lab.index]
lab['y_e'] = lab[7].map(E2I)
lab['y_p'] = lab[5].map(lambda p: P2I.get(p, -1))
lab = lab[lab.y_e.notna()]

for d in (train_all, ev):
    d['w_cert'] = d['clip4'].map(lambda c: CERT_WEIGHTS.get(str(ann.at[c, 8]), 1.0))
    d['unc_B'] = d['clip4'].map(lambda c: str(ann.at[c, 8]))
print(f"train {len(train_all)} | {EVAL_SPLIT} {len(ev)} | folds: "
      f"{[sorted(s for s in train_sources if FOLD_OF[s] == k) for k in range(N_FOLDS)]}")
print(f"forecaster inner-dev episodes: {inner_dev_sources}")

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## Role-grounded face features (from G6b, clips I–III only)

In [ ]:
ROLE = pd.read_csv(ROLE_CSV).set_index('sample_id')
need = set(train_all.sample_id) | set(ev.sample_id)
miss = need - set(ROLE.index)
assert not miss, f"{len(miss)} MCIS without face features, e.g. {sorted(miss)[:3]}"
FACE_ROLES = {'full': ['A', 'listener', 'listener_ctx', 'ctx_II', 'ctx_I'],
              'early': ['A', 'listener_early', 'listener_ctx', 'ctx_II', 'ctx_I']}
FACE_ROLES['presence'] = FACE_ROLES['full']


def face_cols(fset):
    dims = [10, 11] if fset == 'presence' else range(12)
    return [f"{r}_{i}" for r in FACE_ROLES[fset] for i in dims]


def face_matrix(d, fset):
    return ROLE.loc[d.sample_id, face_cols(fset)].values.astype(np.float32)


# z-score with statistics of the training episodes only
FSTAT = {}
for fset in FACE_ROLES:
    X = face_matrix(train_all, fset)
    FSTAT[fset] = (X.mean(0), X.std(0) + 1e-6)

vis = ROLE.loc[ev.sample_id, 'listener_10'].values > 0
early = ROLE.loc[ev.sample_id, 'listener_early_10'].values > 0
bctx = ROLE.loc[ev.sample_id, 'listener_ctx_10'].values > 0
print(f"{EVAL_SPLIT}: listener visible {vis.mean() * 100:.1f}% | in first 80% of III {early.mean() * 100:.1f}% | "
      f"listener also seen in I/II {bctx.mean() * 100:.1f}%")


class FaceForecaster(nn.Module):
    # B1 (raw clip encoder over I-III) + a role-face vector added to the pooled representation.
    # The face MLP's last layer is zero-initialised, so the model starts exactly as B1.

    def __init__(self, n_face, d=512):
        super().__init__()
        self.base = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, clip_idx, rec, face):
        b = self.base
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
        tok = b.enc(flat).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(face))

## Forecasters under both selection protocols

In [ ]:
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


fc_train = train_all[~train_all.source_folder.isin(inner_dev_sources)].reset_index(drop=True)
fc_dev = train_all[train_all.source_folder.isin(inner_dev_sources)].reset_index(drop=True)
_T = {}


def make_T(rows_name, d, fset):
    key = (rows_name, fset)
    if key not in _T:
        idx = torch.tensor([[CIDX[c] for c in r] for r in d[['clip1', 'clip2', 'clip3']].values], device=DEVICE)
        rec = torch.zeros(len(d), 3, N_REC, device=DEVICE)
        if fset is None:
            face = None
        else:
            mu, sd = FSTAT[fset]
            face = torch.tensor((face_matrix(d, fset) - mu) / sd, device=DEVICE)
        _T[key] = (idx, rec, torch.tensor(d.yB.values, device=DEVICE), face)
    return _T[key]


def run(model, T, i, j):
    return model(T[0][i:j], T[1][i:j]) if T[3] is None else model(T[0][i:j], T[1][i:j], T[3][i:j])


def fc_predict(model, T, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(T[0]), bs):
            out.append(F.softmax(run(model, T, i, i + bs), -1).cpu())
    return torch.cat(out).numpy()


def train_forecaster(fset, protocol, seed):
    seed_all(seed)
    tr_rows, tr_name = (train_all, 'train_all') if protocol == 'val' else (fc_train, 'fc_train')
    T_tr = make_T(tr_name, tr_rows, fset)
    T_ev = make_T('ev', ev, fset)
    T_sel = T_ev if protocol == 'val' else make_T('fc_dev', fc_dev, fset)
    model = (Forecaster(use_raw=True, use_traj=False) if fset is None
             else FaceForecaster(T_tr[3].shape[1])).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    y = T_tr[2]
    y_sel = T_sel[2].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(FC_EPOCHS):
        model.train()
        perm = torch.randperm(len(y), device=DEVICE)
        for i in range(0, len(perm), FC_BATCH):
            j = perm[i:i + FC_BATCH]
            Tj = tuple(t[j] if t is not None else None for t in T_tr)
            loss = F.cross_entropy(run(model, Tj, 0, len(j)), y[j])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sel_uar = war_uar(fc_predict(model, T_sel).argmax(1), y_sel, 7)[1]
        if sel_uar > best:
            best, bad = sel_uar, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return fc_predict(model, T_ev), best


yB = ev.yB.values
src = ev.source_folder.values
results, PROBS = [], {}
for name, fset, protocol in EXPERIMENTS:
    PROBS[name] = []
    for seed in SEEDS:
        p, sel = train_forecaster(fset, protocol, seed)
        PROBS[name].append(p)
        w, u = war_uar(p.argmax(1), yB, 7)
        wv, uv = war_uar(p[vis].argmax(1), yB[vis], 7)
        results.append({'exp': name, 'protocol': protocol, 'seed': seed, 'sel_UAR': sel, 'UAR': u, 'WAR': w,
                        'UAR_listener_visible': uv, 'WAR_listener_visible': wv})
        print({k: round(v, 2) if isinstance(v, float) else v for k, v in results[-1].items()}, flush=True)
    torch.cuda.empty_cache()

res = pd.DataFrame(results)
res.to_csv(f"{OUT_DIR}/g7b_results_per_seed.csv", index=False)
np.savez(f"{OUT_DIR}/g7b_{EVAL_SPLIT}_probs.npz", sample_id=ev.sample_id.values,
         **{n.replace('@', '_at_').replace('+', '_'): np.stack(v) for n, v in PROBS.items()})
print("\n== mean ± std over seeds ==")
print(res.drop(columns=['seed', 'protocol']).groupby('exp', sort=False).agg(['mean', 'std']).round(2).to_string())
for prot, ref in G3B_B1_SEED_MEAN.items():
    got = res[(res.exp == ('B1' if prot == 'inner_dev' else 'B1@val'))].UAR.mean()
    print(f"reproduction check, B1 ({prot}): seed-mean UAR {got:.2f} vs G3b {ref:.2f}")

## Paired comparisons against B1, subsets and the gate

In [ ]:
def paired_diff_ci(pa, pb, y, src, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        wa, ua = war_uar(pa[idx], y[idx], 7)
        wb, ub = war_uar(pb[idx], y[idx], 7)
        d.append((ua - ub, wa - wb))
    return np.percentile(np.array(d), [2.5, 97.5], axis=0)


ENS = {n: np.mean(v, 0).argmax(1) for n, v in PROBS.items()}
SUBSETS = {'all': np.ones(len(yB), bool), 'listener visible': vis, 'listener not visible': ~vis,
           'listener in first 80% of III': early, 'listener also seen in I/II': bctx}
PAIRS = [("B1+faces", "B1"), ("B1+faces_early", "B1"), ("B1+presence", "B1"),
         ("B1+faces@val", "B1@val"), ("B1+faces_early@val", "B1@val"), ("B1+presence@val", "B1@val")]
verdict = {}
for sname, m in SUBSETS.items():
    print(f"\n== {sname}: n={m.sum()} ==")
    if m.sum() < 30:
        print("  too few MCIS, skipped")
        continue
    for n in ENS:
        report(f"  {n}", ENS[n][m], yB[m], src[m])
    for a, b in PAIRS:
        lo, hi = paired_diff_ci(ENS[a][m], ENS[b][m], yB[m], src[m])
        wa, ua = war_uar(ENS[a][m], yB[m], 7); wb, ub = war_uar(ENS[b][m], yB[m], 7)
        col = 'UAR' if sname == 'all' else ('UAR_listener_visible' if sname == 'listener visible' else None)
        wins = ""
        if col:
            pa = res[res.exp == a].set_index('seed')[col]; pb = res[res.exp == b].set_index('seed')[col]
            wins = f"({int(((pa - pb) > 0).sum())}/{len(SEEDS)} seeds)"
            verdict[(sname, a)] = (ua - ub, lo[0], hi[0], int(((pa - pb) > 0).sum()))
        print(f"  {a:<20} - {b:<7} ΔUAR {ua - ub:+5.2f} [{lo[0]:+5.2f},{hi[0]:+5.2f}] {wins}  "
              f"ΔWAR {wa - wb:+5.2f} [{lo[1]:+5.2f},{hi[1]:+5.2f}]")

d1 = verdict[('all', 'B1+faces')]
d2 = verdict[('all', 'B1+faces@val')]
dp = verdict[('all', 'B1+presence')]
print("\n== GATE ==")
print(f"1. B1+faces vs B1 (inner_dev): ΔUAR {d1[0]:+.2f} [{d1[1]:+.2f},{d1[2]:+.2f}], {d1[3]}/{len(SEEDS)} seed wins -> "
      f"{'PASS' if d1[1] > 0 and d1[3] >= 3 else ('WEAK (positive, CI includes 0)' if d1[0] > 0 else 'FAIL')}")
print(f"2. B1+faces@val vs B1@val: ΔUAR {d2[0]:+.2f} -> {'PASS' if d2[0] >= 0 else 'FAIL'}")
print(f"3. presence-only control (inner_dev): ΔUAR {dp[0]:+.2f} -> "
      f"{'OK (below faces)' if dp[0] < d1[0] else 'CAUTION: presence alone explains the gain'}")
print("4. listener-visible subset and early-frame arm: see tables above (descriptive)")